# 08 - Silver 3: secondary events and the enriched join
`silver_network_enriched` = events + current status (CDC) + secondary-event summary + region/tower attributes.
Secondary events are summarised per measurement_id first, so the join can never multiply rows.

### Settings (names of your catalog and schema)

In [0]:
from pyspark.sql import functions as F, Window as W

BRONZE = "capstone6_telecom.bronze"
SILVER = "capstone6_telecom.silver"
GOLD   = "capstone6_telecom.gold"
cat    = SILVER   # this notebook writes to the silver schema

EXTREME_LATENCY_MS = 500                      # TC02: exactly 35 rows are above this
EVENT_STATUS = ["OPEN", "FAILED", "PENDING", "COMPLETED"]
CDC_STATUS   = EVENT_STATUS + ["CANCELLED", "REVERSED"]
SEC_TYPES    = ["CHECK", "ALERT", "REVIEW"]
OUTAGE_STATUS = "FAILED"                      # a reading counts as an outage when its CURRENT status is this

### Helper functions (just run this cell)

In [0]:
def failed_rules(*pairs):
    """pairs = (condition_meaning_the_rule_FAILED, rule_id). Returns an array of failed rule ids."""
    return F.filter(F.array(*[F.when(c, F.lit(r)) for c, r in pairs]), lambda x: x.isNotNull())

def not_in(col, values):
    """True when the value is null or not in the list."""
    return F.coalesce(~F.col(col).isin(values), F.lit(True))

def rejected_rows(df, record_type, key_col, raw_cols):
    """df needs: rule_id, action, source_file, ingest_ts, key_col and raw_cols."""
    rules = spark.table(f"{cat}.silver_dq_rules").select("rule_id", F.col("description").alias("reason"))
    return (df.join(F.broadcast(rules), "rule_id", "left")
              .select(F.lit(record_type).alias("record_type"),
                      F.col(key_col).cast("string").alias("record_key"),
                      "rule_id", "action", "reason", "source_file",
                      F.col("ingest_ts").alias("bronze_ingest_ts"),
                      F.to_json(F.struct(*raw_cols)).alias("raw_record"),
                      F.current_timestamp().alias("rejected_at")))

def write_rejected(df, record_type):
    """Idempotent: replaces only this record_type's rows, so re-runs never duplicate."""
    t = f"{cat}.silver_rejected_records"
    if spark.catalog.tableExists(t):
        (df.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"record_type = '{record_type}'").saveAsTable(t))
    else:
        df.write.format("delta").saveAsTable(t)

### Function: build Silver secondary events (just run this cell)

In [0]:
def build_silver_secondary(b, events, regions):
    """Bronze secondary events -> (clean, rejected)."""
    raw_cols = b.columns
    s = (b
         .withColumn("event_ts", F.expr("try_to_timestamp(event_timestamp)"))
         .withColumn("value_n", F.expr("try_cast(event_value AS DOUBLE)"))
         .withColumn("type_c", F.upper(F.trim(F.col("event_type"))))
         .join(F.broadcast(events.select("measurement_id").distinct().withColumn("_evt_ok", F.lit(True))), "measurement_id", "left")
         .join(F.broadcast(regions.select(F.col("region_id").alias("related_entity_id")).distinct().withColumn("_reg_ok", F.lit(True))),
               "related_entity_id", "left"))

    w = W.partitionBy("event_id").orderBy(F.col("ingest_ts").desc(), F.col("source_file").desc())
    s = (s.withColumn("rn", F.row_number().over(w))
          .withColumn("is_dup", F.col("event_id").isNotNull() & (F.col("rn") > 1))
          .withColumn("failed", failed_rules(
              (F.col("event_id").isNull(), "DQ16"),
              (F.col("_evt_ok").isNull(), "DQ17"),
              (not_in("type_c", SEC_TYPES), "DQ18"),
              (F.col("value_n").isNull(), "DQ19"),
              (F.col("event_ts").isNull(), "DQ20"),
              (F.col("_reg_ok").isNull(), "DQ21"))))

    clean = (s.filter(~F.col("is_dup") & (F.size("failed") == 0))
              .select("event_id", "measurement_id", "event_ts",
                      F.col("related_entity_id").alias("related_region_id"),
                      F.col("type_c").alias("event_type"), F.col("value_n").alias("event_value"),
                      "source_system", "source_file", F.col("ingest_ts").alias("bronze_ingest_ts")))
    rej_fail = (s.filter(~F.col("is_dup") & (F.size("failed") > 0))
                 .withColumn("rule_id", F.explode("failed")).withColumn("action", F.lit("REJECTED")))
    rej_dup = s.filter(F.col("is_dup")).withColumn("rule_id", F.lit("DQ16")).withColumn("action", F.lit("DEDUPED"))
    rejected = rejected_rows(rej_fail.unionByName(rej_dup), "SECONDARY", "event_id", raw_cols)
    return clean, rejected

In [0]:
events  = spark.table(f"{cat}.silver_network_events")
regions = spark.table(f"{cat}.silver_dim_region")
raw     = spark.table(f"{BRONZE}.bronze_secondary_events")

clean, rejected = build_silver_secondary(raw, events, regions)
clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{cat}.silver_secondary_events")
write_rejected(rejected, "SECONDARY")

In [0]:
sec = spark.table(f"{cat}.silver_secondary_events")
sec_sum = (sec.groupBy("measurement_id").agg(
              F.count("*").alias("secondary_event_count"),
              (F.max(F.when(F.col("event_type") == "ALERT", 1).otherwise(0)) == 1).alias("has_alert"),
              F.max("event_value").alias("max_secondary_value"),
              F.expr("max_by(event_type, event_ts)").alias("latest_secondary_type")))

cur = spark.table(f"{cat}.silver_network_current").select(
        "measurement_id", "current_status",
        F.col("last_sequence").alias("status_sequence"), F.col("last_change_ts").alias("status_change_ts"),
        F.col("last_change_reason").alias("status_change_reason"))

reg = spark.table(f"{cat}.silver_dim_region").select(
        "region_id", F.col("region_group").alias("region_group"), F.col("risk_tier").alias("region_risk_tier"))
twr = spark.table(f"{cat}.silver_dim_tower").select(
        "tower_id", F.col("region_group").alias("tower_region_group"), F.col("risk_tier").alias("tower_risk_tier"),
        F.col("is_active").alias("tower_is_active"))

enriched = (events
            .join(cur, "measurement_id", "left")
            .join(sec_sum, "measurement_id", "left")
            .join(F.broadcast(reg), "region_id", "left")
            .join(F.broadcast(twr), "tower_id", "left")
            .fillna({"secondary_event_count": 0, "has_alert": False}))

enriched.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{cat}.silver_network_enriched")

## Checks: no row loss or fan-out

In [0]:
display(spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {BRONZE}.bronze_secondary_events) AS bronze_secondary,           -- 310
       (SELECT COUNT(*) FROM {cat}.silver_secondary_events) AS silver_secondary,           -- 310
       (SELECT COUNT(*) FROM {cat}.silver_rejected_records WHERE record_type='SECONDARY') AS rejected, -- 0
       (SELECT COUNT(*) FROM {cat}.silver_network_events) AS events,                       -- 11,500
       (SELECT COUNT(*) FROM {cat}.silver_network_enriched) AS enriched,                   -- must equal events
       (SELECT COUNT(*) FROM {cat}.silver_network_enriched WHERE secondary_event_count > 0) AS events_with_secondary, -- 310
       (SELECT COUNT(*) FROM {cat}.silver_network_enriched WHERE current_status IS NULL) AS missing_current_status   -- 0"""))

display(spark.sql(f"SELECT event_type, COUNT(*) AS rows FROM {cat}.silver_secondary_events GROUP BY 1"))